# CSC3310 Final Project - A* Search Algorithm

In [1]:
import json
import math
import os

In [2]:
class Graph:
    def __init__(self):
        self.nodes = {}
        self.adjacency = {}

    def add_node(self, node_id, coordinates):
        self.nodes[node_id] = coordinates
        self.adjacency[node_id] = []

    def add_edge(self, from_id, to_id, distance_meters, transport_modes, has_stairs=False, bidirectional=True):
        edge = {
            "from": from_id,
            "to": to_id,
            "distance_meters": distance_meters,
            "transport_modes": transport_modes,
            "has_stairs": has_stairs,
            "bidirectional": bidirectional,
        }
        self.adjacency[from_id].append(edge)
        self.adjacency[to_id].append(edge)

In [3]:
def haversine_distance(a, b):
    lon1, lat1, lon2, lat2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371000 * math.asin(math.sqrt(h))

In [4]:
def find_route(graph, start_node_id, end_node_id, options):
    start_node = graph.nodes.get(start_node_id)
    end_node = graph.nodes.get(end_node_id)

    if not start_node or not end_node:
        return None
    if start_node_id == end_node_id:
        return [start_node]

    open_set = {}
    closed_set = set()
    came_from = {}
    g_scores = {}

    g_scores[start_node_id] = 0
    open_set[start_node_id] = {
        "g_score": 0,
        "f_score": haversine_distance(start_node, end_node),
    }

    while open_set:
        current_id = get_lowest_f_score(open_set)

        if current_id == end_node_id:
            return reconstruct_path(graph, came_from, current_id)

        del open_set[current_id]
        closed_set.add(current_id)

        current_g = g_scores.get(current_id, math.inf)

        for edge in graph.adjacency[current_id]:
            if options["transport_mode"] not in edge["transport_modes"]:
                continue

            if options.get("no_stairs") and edge["has_stairs"]:
                continue

            if not edge["bidirectional"] and edge["to"] == current_id:
                continue

            neighbor_id = edge["to"] if edge["from"] == current_id else edge["from"]

            if neighbor_id in closed_set:
                continue

            neighbor_node = graph.nodes.get(neighbor_id)
            if not neighbor_node:
                continue

            tentative_g = current_g + edge["distance_meters"]
            known_g = g_scores.get(neighbor_id, math.inf)

            if tentative_g < known_g:
                came_from[neighbor_id] = current_id
                g_scores[neighbor_id] = tentative_g

                h = haversine_distance(neighbor_node, end_node)

                open_set[neighbor_id] = {
                    "g_score": tentative_g,
                    "f_score": tentative_g + h,
                }

    return None


def get_lowest_f_score(open_set):
    lowest_id = ""
    lowest_score = math.inf

    for id, node in open_set.items():
        if node["f_score"] < lowest_score:
            lowest_score = node["f_score"]
            lowest_id = id

    return lowest_id


def reconstruct_path(graph, came_from, end_node_id):
    path = []
    current = end_node_id

    while current in came_from:
        path.insert(0, graph.nodes[current])
        current = came_from[current]

    path.insert(0, graph.nodes[current])

    return path

In [5]:
def load_demo_data(path="demo-data.json"):
    """Load the compact demo graph and build the Graph used by A*."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Demo data not found: {os.path.abspath(path)}")

    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    graph = Graph()
    for node_id, node in data["nodes"].items():
        graph.add_node(node_id, node["coordinates"])

    for edge in data["edges"].values():
        graph.add_edge(
            edge["from"],
            edge["to"],
            edge["distanceMeters"],
            edge["transportModes"],
            edge.get("hasStairs", False),
            edge.get("bidirectional", True),
        )

    return data, graph


In [6]:
demo, demo_graph = load_demo_data()
print(demo["name"])
print(f"Nodes: {len(demo['nodes'])}  Edges: {len(demo['edges'])}  POIs: {len(demo['pointsOfInterest'])}")

start = demo["demoStart"]
end = demo["demoEnd"]
route = find_route(
    demo_graph,
    start,
    end,
    {"transport_mode": "walking", "no_stairs": True},
)

if route:
    print(f"Route found from {start} to {end}: {len(route)} points")
    print("Start:", route[0])
    print("End:", route[-1])
else:
    print("No route found")

print("POIs:")
for poi in demo["pointsOfInterest"]:
    print("-", poi["properties"].get("name", poi["properties"]["id"]))


GLCC Demo Dataset
Nodes: 24  Edges: 23  POIs: 2
Route found from node-773 to node-751: 14 points
Start: [-89.0183716, 43.816059]
End: [-89.0170135, 43.8153925]
POIs:
- Kern Lodge
- Parking Lot 1
